In [2]:
import numpy as np
import cv2
print(np.linspace(0,10,5))
print(cv2.ocl.haveOpenCL())  # Check before enabling

[ 0.   2.5  5.   7.5 10. ]
True


In [4]:
import cv2              #Used for Image Processing 
import mediapipe as mp  #Googles framework trained on data like hands face body etc
import pygame           #Pygame is used to handle Audio Features like Playing sounds
import numpy as np      #Numerical Python for handling number data
import threading        #Used for Concurrent execution of process to make everything smooth
import math             #Inbuilt python library for handling mathematical features
import time             #Used to calculate time and latency

                           #Speed optimizations
cv2.setNumThreads(8)       #Used or divides the work into cores my intel i5 is octa core processor for mediapipe and pygame
cv2.ocl.setUseOpenCL(True) #Open Computing Langauage framework for GPu Procesing
#Audio Department
pygame.mixer.pre_init(44100,-16,2,512)    #Slightly larger buffer for smoother audio stream
pygame.mixer.init()                       #44000 is frequency standad, - means signed and determines dynamic range,audio channel mono or stereo 
pygame.mixer.set_num_channels(64)         #512 is the buffer for storing audio low=zero lag, high=smooth
                                          #64number of sounds played simultaneously else it cuts
#Initialization command for hardware sound card setting or applying these values to soundcard
#Generating the Piano sound using function
def generate_piano(freq,duration=2.0):
    sample_rate=44100
    t=np.linspace(0,duration,int(sample_rate*duration),False)
    #linspace to evenly make sequence of numbers, inside the numpy library
    #Start and stop (sample_rate*durration is the total number=88200 numbers total)

    #Piano harmonics generation
    #1.0 =1.0,0.5..etc are volume
    #sin is a sin wave generator s curve shape high and low 
    #sin wave is treated as circle so 2pi is used
    #freq x t is done to calculate the vibration at each timestamps for 2 seconds
    wave=(1.0*np.sin(2*np.pi*freq*t)+
          0.5*np.sin(2*np.pi*(2.001*freq)*t)+ 
          0.2*np.sin(2*np.pi*(3.002*freq)*t)+
          0.1*np.sin(2*np.pi*(4.003*freq)*t))
    attack_samples=int(0.01*sample_rate) #Start the note smoothly
    envelope=np.ones_like(t)             #Contain the notes inner sounds
    envelope[:attack_samples]=np.linspace(0,1,attack_samples)
    decay=np.exp(-2.5*t)                 #For a realistic sound ending
    processed_wave=wave*envelope*decay      
    max_val=np.max(np.abs(processed_wave))
    if max_val>0:
        processed_wave=(processed_wave/max_val)*0.6 #Headroom to prevent crackle
    samples=(processed_wave*32767).astype(np.int16) #Saving for 16 bit audio lrgst possbile is 32767 as whole numbers int
    stereo_samples=np.ascontiguousarray(np.column_stack((samples, samples))) #for both stereo duplication, ascontigous array stores 
                                                                             #in memory as a non broken line in ram
    return pygame.sndarray.make_sound(stereo_samples) #Produces sound usind make_sound and connects to pygame
#Labelling the keys with frequncy and white or black True 
PIANO_DATA=[
    ("C4",261.63,False),("C#4",277.18,True),("D4",293.66,False),
    ("D#4",311.13,True),("E4",329.63,False),("F4",349.23,False),
    ("F#4",369.99,True),("G4",392.00,False),("G#4",415.30,True),
    ("A4",440.00,False),("A#4",466.16,True),("B4",493.88,False),
    ("C5",523.25,False),("C#5",554.37,True),("D5",587.33,False),
    ("D#5",622.25,True),("E5",659.25,False),("F5",698.46,False),
    ("F#5",739.99,True),("G5",783.99,False),("G#5",830.61,True),
    ("A5",880.00,False),("A#5",932.33,True),("B5",987.77,False),
    ("C6",1046.50,False)]
WIDTH,HEIGHT=1005,620 
PIANO_HEIGHT=270           
WHITE_TRIGGER_LINE=int(PIANO_HEIGHT*0.65) 
BLACK_KEY_VISUAL_HEIGHT=int(PIANO_HEIGHT*0.53)
total_white_keys=sum(1 for _,_,s in PIANO_DATA if not s)
WHITE_KEY_W=WIDTH//total_white_keys
PRESS_THRESHOLD_Z=-0.060 
RELEASE_THRESHOLD_Z=-0.030
#Landmarks for strict straight logic
FINGERTIPS=[8,12,16,20]     
FINGER_PIP_IDS=[6,10,14,18] #Middle joints #Thumb avoided as it is hard to press in air
FINGER_BASE_IDS=[5,9,13,17] #Knuckles
white_keys, black_keys, key_sounds=[],[],[]
for _,freq,_ in PIANO_DATA:key_sounds.append(generate_piano(freq))
w_idx = 0
for i,(_,_,is_sharp) in enumerate(PIANO_DATA):
    key={'id':i,'is_pressed':False}
    if not is_sharp:
        key['rect']=(w_idx*WHITE_KEY_W,0,(w_idx+1)*WHITE_KEY_W,PIANO_HEIGHT)
        white_keys.append(key);w_idx+=1
    else:
        prev=w_idx*WHITE_KEY_W
        bw=int(WHITE_KEY_W*0.75)
        key['rect']=(prev-bw//2,0,prev+bw//2,BLACK_KEY_VISUAL_HEIGHT)
        black_keys.append(key)
all_keys=white_keys+black_keys
class CameraStream:
    def __init__(self):
        self.cap=cv2.VideoCapture(0)
        self.cap.set(cv2.CAP_PROP_FRAME_WIDTH,640)
        self.cap.set(cv2.CAP_PROP_FRAME_HEIGHT,480)
        self.cap.set(cv2.CAP_PROP_BUFFERSIZE,1)
        self.success,self.frame=self.cap.read()
        self.stopped=False
    def start(self):
        threading.Thread(target=self.update,daemon=True).start()
        return self
    def update(self):
        while not self.stopped:
            ret,frame=self.cap.read()
            if ret:self.frame=frame
    def read(self):return self.frame
    def stop(self):self.stopped=True;self.cap.release()
stream=CameraStream().start()
mp_hands=mp.solutions.hands
hands=mp_hands.Hands(
    static_image_mode=False, 
    max_num_hands=2, 
    min_detection_confidence=0.6, 
    model_complexity=0
)
while True:
    raw_frame=stream.read()
    if raw_frame is None:continue
    frame=cv2.flip(raw_frame,1)
    frame=cv2.resize(frame,(WIDTH,HEIGHT))
    results=hands.process(cv2.cvtColor(cv2.resize(frame,(480, 240)),cv2.COLOR_BGR2RGB))
    keys_currently_touched={}
    if results.multi_hand_landmarks:
        for hand_lms in results.multi_hand_landmarks:
            for i in range(len(FINGERTIPS)):
                tip=hand_lms.landmark[FINGERTIPS[i]]
                pip=hand_lms.landmark[FINGER_PIP_IDS[i]]
                base=hand_lms.landmark[FINGER_BASE_IDS[i]]           
                #Vertical check:Tip above middle, middle above Knuckle
                is_straight_v = tip.y < pip.y < base.y                
                #Distance check:Ensure finger isn't curled towards the palm
                d_tip_base=math.hypot(tip.x-base.x,tip.y-base.y)
                d_pip_base=math.hypot(pip.x-base.x,pip.y-base.y)
                is_fully_extended=d_tip_base>(d_pip_base*1.15)        
                is_active=is_straight_v and is_fully_extended            
                cx,cy,cz=tip.x*WIDTH,tip.y*HEIGHT,tip.z
                hit_id=None
                #Only trigger if the finger is FULLY OPEN and STRAIGHT
                if is_active:
                    for k in black_keys:
                        x1,y1,x2,y2=k['rect']
                        if x1<cx<x2 and y1<cy<y2:
                            hit_id=k['id']
                            break 
                    if hit_id is None:
                        for k in white_keys:
                            x1,y1,x2,y2=k['rect']
                            trigger_margin=(x2-x1)*0.1 
                            if (x1+trigger_margin)<cx<(x2-trigger_margin)and \
                               WHITE_TRIGGER_LINE<cy<PIANO_HEIGHT:
                                hit_id=k['id']
                                break
                if hit_id is not None:
                    if hit_id not in keys_currently_touched or cz < keys_currently_touched[hit_id]:
                        keys_currently_touched[hit_id]=cz      
                color=(255,255,0) if is_active else (0,0,0)
                cv2.circle(frame,(int(cx),int(cy)),8,color,-1)
    for k in all_keys:
        kid=k['id']
        current_z=keys_currently_touched.get(kid,1.0)
        if not k['is_pressed']:
            if current_z < PRESS_THRESHOLD_Z:
                pygame.mixer.find_channel(True).play(key_sounds[kid],fade_ms=50)
                k['is_pressed']=True
        else:
            if current_z > RELEASE_THRESHOLD_Z:
                k['is_pressed']=False
    #Draw
    cv2.line(frame,(0, WHITE_TRIGGER_LINE),(WIDTH, WHITE_TRIGGER_LINE),(0, 0, 255),2) 
    for k in white_keys:
        x1,y1,x2,y2=k['rect']
        clr=(0,255,120) if k['is_pressed'] else (255,255,255)
        cv2.rectangle(frame,(x1, y1),(x2, y2),clr,-1 if k['is_pressed'] else 2)
        cv2.rectangle(frame,(x1, y1),(x2, y2),(200,200,200),1)
    for k in black_keys:
        x1,y1,x2,y2=k['rect']
        clr=(0,180,80) if k['is_pressed'] else (20,20,20)
        cv2.rectangle(frame,(x1, y1),(x2, y2),clr,-1)
    cv2.imshow("YAMAHA-MONATAGE-M-SYNTH", frame)
    if cv2.waitKey(1)&0xFF in[ord('q'), 27]: break
stream.stop()
cv2.destroyAllWindows()


In [2]:
import cv2              #Used for Image Processing 
import mediapipe as mp  #Googles framework trained on data like hands face body etc
import pygame           #Pygame is used to handle Audio Features like Playing sounds
import numpy as np      #Numerical Python for handling number data
import threading        #Used for Concurrent execution of process to make everything smooth
import math             #Inbuilt python library for handling mathematical features
import time             #Used to calculate time and latency

                           #Speed optimizations
cv2.setNumThreads(8)       #Used or divides the work into cores my intel i5 is octa core processor for mediapipe and pygame
cv2.ocl.setUseOpenCL(True) #Open Computing Langauage framework for GPu Procesing

#Audio Department
pygame.mixer.pre_init(44100,-16,2,512)    #Slightly larger buffer for smoother audio stream
pygame.mixer.init()                       #44000 is frequency standad, - means signed and determines dynamic range,audio channel mono or stereo 
pygame.mixer.set_num_channels(64)         #512 is the buffer for storing audio low=zero lag, high=smooth
                                          #64number of sounds played simultaneously else it cuts
#Initialization command for hardware sound card setting or applying these values to soundcard

#Generating the Piano sound using function
def generate_piano(freq,duration=2.0):
    sample_rate=44100
    t=np.linspace(0,duration,int(sample_rate*duration),False)
    wave=(1.0*np.sin(2*np.pi*freq*t)+
          0.5*np.sin(2*np.pi*(2.001*freq)*t)+ 
          0.2*np.sin(2*np.pi*(3.002*freq)*t)+
          0.1*np.sin(2*np.pi*(4.003*freq)*t))
    attack_samples=int(0.01*sample_rate)
    envelope=np.ones_like(t)
    envelope[:attack_samples]=np.linspace(0,1,attack_samples)
    decay=np.exp(-2.5*t)
    processed_wave=wave*envelope*decay      
    max_val=np.max(np.abs(processed_wave))
    if max_val>0:
        processed_wave=(processed_wave/max_val)*0.6
    samples=(processed_wave*32767).astype(np.int16)
    stereo_samples=np.ascontiguousarray(np.column_stack((samples, samples)))
    return pygame.sndarray.make_sound(stereo_samples)

#Labelling the keys with frequncy and white or black True 
PIANO_DATA=[
    ("C4",261.63,False),("C#4",277.18,True),("D4",293.66,False),
    ("D#4",311.13,True),("E4",329.63,False),("F4",349.23,False),
    ("F#4",369.99,True),("G4",392.00,False),("G#4",415.30,True),
    ("A4",440.00,False),("A#4",466.16,True),("B4",493.88,False),
    ("C5",523.25,False),("C#5",554.37,True),("D5",587.33,False),
    ("D#5",622.25,True),("E5",659.25,False),("F5",698.46,False),
    ("F#5",739.99,True),("G5",783.99,False),("G#5",830.61,True),
    ("A5",880.00,False),("A#5",932.33,True),("B5",987.77,False),
    ("C6",1046.50,False)]

WIDTH,HEIGHT=1005,620 
PIANO_HEIGHT=270           
WHITE_TRIGGER_LINE=int(PIANO_HEIGHT*0.65) 
BLACK_KEY_VISUAL_HEIGHT=int(PIANO_HEIGHT*0.53)
total_white_keys=sum(1 for _,_,s in PIANO_DATA if not s)
WHITE_KEY_W=WIDTH//total_white_keys
PRESS_THRESHOLD_Z=-0.060 
RELEASE_THRESHOLD_Z=-0.030

#Landmarks for strict straight logic
FINGERTIPS=[8,12,16,20]     
FINGER_PIP_IDS=[6,10,14,18] #Middle joints
FINGER_BASE_IDS=[5,9,13,17] #Knuckles

white_keys, black_keys, key_sounds=[],[],[]
for _,freq,_ in PIANO_DATA: key_sounds.append(generate_piano(freq))
w_idx = 0
for i,(_,_,is_sharp) in enumerate(PIANO_DATA):
    key = {'id':i, 'is_pressed':False}
    if not is_sharp:
        key['rect'] = (w_idx*WHITE_KEY_W, 0, (w_idx+1)*WHITE_KEY_W, PIANO_HEIGHT)
        white_keys.append(key); w_idx += 1
    else:
        prev = w_idx*WHITE_KEY_W
        bw = int(WHITE_KEY_W*0.75)
        key['rect'] = (prev-bw//2, 0, prev+bw//2, BLACK_KEY_VISUAL_HEIGHT)
        black_keys.append(key)
all_keys = white_keys + black_keys

class CameraStream:
    def __init__(self):
        self.cap = cv2.VideoCapture(0)
        self.cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
        self.cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)
        self.cap.set(cv2.CAP_PROP_BUFFERSIZE, 1)
        self.success, self.frame = self.cap.read()
        self.stopped = False
    def start(self):
        threading.Thread(target=self.update, daemon=True).start()
        return self
    def update(self):
        while not self.stopped:
            ret, frame = self.cap.read()
            if ret: self.frame = frame
    def read(self): return self.frame
    def stop(self): self.stopped = True; self.cap.release()

stream = CameraStream().start()

mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils   # Used to draw connections between landmarks

hands = mp_hands.Hands(
    static_image_mode=False, 
    max_num_hands=2, 
    min_detection_confidence=0.6, 
    model_complexity=0
)

while True:
    raw_frame = stream.read()
    if raw_frame is None: continue
    frame = cv2.flip(raw_frame, 1)
    frame = cv2.resize(frame, (WIDTH, HEIGHT))
    results = hands.process(cv2.cvtColor(cv2.resize(frame, (480, 240)), cv2.COLOR_BGR2RGB))
    keys_currently_touched = {}

    if results.multi_hand_landmarks:
        for hand_lms in results.multi_hand_landmarks:
            for i in range(len(FINGERTIPS)):
                tip = hand_lms.landmark[FINGERTIPS[i]]
                pip = hand_lms.landmark[FINGER_PIP_IDS[i]]
                base = hand_lms.landmark[FINGER_BASE_IDS[i]]           
                
                # --- STRICT STRAIGHT LOGIC ---
                # 1. Vertical check: Tip above Middle (PIP), Middle above Knuckle (Base)
                is_straight_v = tip.y < pip.y < base.y                
                
                # 2. Distance check: Ensure finger isn't curled toward the palm
                # Euclidean distance Tip-to-Base must be significantly longer than PIP-to-Base
                d_tip_base = math.hypot(tip.x - base.x, tip.y - base.y)
                d_pip_base = math.hypot(pip.x - base.x, pip.y - base.y)
                is_fully_extended = d_tip_base > (d_pip_base * 1.15)        
                is_active = is_straight_v and is_fully_extended            
                
                cx, cy, cz = tip.x*WIDTH, tip.y*HEIGHT, tip.z
                hit_id = None

                # Only trigger if the finger is FULLY OPEN and STRAIGHT
                if is_active:
                    for k in black_keys:
                        x1, y1, x2, y2 = k['rect']
                        if x1 < cx < x2 and y1 < cy < y2:
                            hit_id = k['id']
                            break 
                    if hit_id is None:
                        for k in white_keys:
                            x1, y1, x2, y2 = k['rect']
                            trigger_margin = (x2 - x1) * 0.1 
                            if (x1 + trigger_margin) < cx < (x2 - trigger_margin) and \
                               WHITE_TRIGGER_LINE < cy < PIANO_HEIGHT:
                                hit_id = k['id']
                                break

                if hit_id is not None:
                    if hit_id not in keys_currently_touched or cz < keys_currently_touched[hit_id]:
                        keys_currently_touched[hit_id] = cz      

                # Visual feedback: Yellow for active/straight, Black dot if curled
                color = (255, 255, 0) if is_active else (0, 0, 0)
                cv2.circle(frame, (int(cx), int(cy)), 8, color, -1)

            # --- HAND LANDMARKS DRAWING ---
            # Draws green joint circles and blue connection lines between all 21 landmarks
            mp_drawing.draw_landmarks(
                frame,                                                                    # Draw on the display frame
                hand_lms,                                                                 # Landmark data for this hand
                mp_hands.HAND_CONNECTIONS,                                                # Lines connecting joints
                mp_drawing.DrawingSpec(color=(0,0,0), thickness=5, circle_radius=3),  # Green joints
                mp_drawing.DrawingSpec(color=(255,255,0), thickness=5)                     # Blue connections
            )

    for k in all_keys:
        kid = k['id']
        current_z = keys_currently_touched.get(kid, 1.0)
        if not k['is_pressed']:
            if current_z < PRESS_THRESHOLD_Z:
                pygame.mixer.find_channel(True).play(key_sounds[kid], fade_ms=50)
                k['is_pressed'] = True
        else:
            if current_z > RELEASE_THRESHOLD_Z:
                k['is_pressed'] = False

    # --- DRAWING ---
    cv2.line(frame, (0, WHITE_TRIGGER_LINE), (WIDTH, WHITE_TRIGGER_LINE), (0, 0, 255), 2) 
    for k in white_keys:
        x1, y1, x2, y2 = k['rect']
        clr = (0,255,120) if k['is_pressed'] else (255,255,255)
        cv2.rectangle(frame, (x1, y1), (x2, y2), clr, -1 if k['is_pressed'] else 2)
        cv2.rectangle(frame, (x1, y1), (x2, y2), (200,200,200), 1)
    for k in black_keys:
        x1, y1, x2, y2 = k['rect']
        clr = (0,180,80) if k['is_pressed'] else (20,20,20)
        cv2.rectangle(frame, (x1, y1), (x2, y2), clr, -1)

    cv2.imshow("Smooth Air Piano - Strict Mode", frame)
    if cv2.waitKey(1) & 0xFF in [ord('q'), 27]: break

stream.stop()
cv2.destroyAllWindows()